# SnapClassify on a free GPU (optional)

The project's main path trains 10 breeds on your own CPU. This notebook trains all 37 breeds on a free GPU instead.
Open it in Google Colab (or Kaggle) with a GPU: **Runtime → Change runtime type → T4 GPU**.

**How to use it:** push your repository to GitHub, set `REPO` in the setup cell and run that cell, then run the
section you need:

| Section | Run it once | What you get |
|---|---|---|
| 1. Train | your training code passes its tests | the trained model, saved to your Google Drive (`MyDrive/snapclassify/models`) |
| 2. Evaluate and compare | your evaluation code passes its tests | `reports/eval.md` and `reports/compare.json`, downloaded to commit |
| 3. Export for the app | your export code passes its tests | `app/model/`, downloaded to commit |

Colab forgets everything when the runtime resets, so section 1 saves the model to your Drive and the setup cell
brings it back in later sessions. The dataset (about 800 MB, CC BY-SA 4.0) downloads into this Colab machine each
session: never commit it or copy it to your repo.


In [ ]:
REPO = "https://github.com/<your-username>/<your-repo>.git"   # your SnapClassify repo (public, or use a token)

import os, shutil, subprocess
if os.path.exists("/content/snapclassify"):
    subprocess.run(["git", "-C", "/content/snapclassify", "pull", "-q"], check=True)   # your latest pushed code
else:
    subprocess.run(["git", "clone", "-q", REPO, "/content/snapclassify"], check=True)
%cd /content/snapclassify
!pip install -q onnx==1.23.1 onnxscript==0.7.2 onnxruntime==1.30.0 "google-genai>=2.25,<3"

from google.colab import drive
drive.mount("/content/drive")
SAVE = "/content/drive/MyDrive/snapclassify"
os.makedirs(SAVE, exist_ok=True)
if os.path.exists(f"{SAVE}/models") and not os.path.exists("models/snapclassify.pt"):
    shutil.copytree(f"{SAVE}/models", "models", dirs_exist_ok=True)       # the model section 1 saved

import torch, torchvision
print(torch.__version__, torchvision.__version__, "· GPU:", torch.cuda.is_available(),
      "· trained model here:", os.path.exists("models/snapclassify.pt"))

## 1. Train

Predict first: what best validation accuracy do you expect after the head-only phase, and after fine-tuning?
Then train: 3 epochs with the backbone frozen, then 3 with the last two blocks fine-tuned, on all 37 breeds.


In [ ]:
!python -m snapclassify.train --data /content/data --epochs-head 3 --epochs-finetune 3 --batch-size 64 --workers 2
shutil.copytree("models", f"{SAVE}/models", dirs_exist_ok=True)
print("saved to", f"{SAVE}/models:", sorted(os.listdir(f"{SAVE}/models")))

## 2. Evaluate and compare

Predict first: the test top-1, and whether Gemini will beat your model on the same photos.
The comparison sends 50 **dataset** photos to the Gemini free tier. Add your key under 🔑 Secrets (left sidebar) as
`GEMINI_API_KEY` and allow this notebook to read it. It waits between calls to stay inside the free tier's limits,
so it takes about six minutes. Then commit the two downloaded reports.


In [ ]:
from google.colab import files, userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
assert os.path.exists("models/snapclassify.pt"), "no trained model: run section 1 first"
!python -m snapclassify.evaluate --data /content/data --compare 50
!zip -qr reports.zip reports
files.download("reports.zip")

## 3. Export for the app

Choose your "not sure" threshold from the table in `reports/eval.md`, set it below, and export. Unzip the download
into your repo and commit `app/model/` (model.onnx, labels.json, config.json): Streamlit Community Cloud serves the
app from your repo.


In [ ]:
THRESHOLD = 0.5   # yours, from the table in reports/eval.md
from google.colab import files
assert os.path.exists("models/snapclassify.pt"), "no trained model: run section 1 first"
!python -m snapclassify.export --threshold $THRESHOLD
!zip -qr app_model.zip app/model
files.download("app_model.zip")